## Orador de Debates Respaldado

## Extractor de Citas


In [14]:
import json
import time
import urllib.request
from html.parser import HTMLParser
from typing import List, Dict

class QuotesHTMLParser(HTMLParser):
    """Parsea el HTML estructurado de la web para extraer citas y autores."""
    def __init__(self):
        super().__init__()
        self.quotes: List[Dict[str, str]] = []
        self.in_quote = False
        self.in_author = False
        self.current_quote = ""
        self.current_author = ""

    def handle_starttag(self, tag: str, attrs: list):
        attrs_dict = dict(attrs)
        if tag == "span" and attrs_dict.get("class") == "text":
            self.in_quote = True
            self.current_quote = ""
        elif tag == "small" and attrs_dict.get("class") == "author":
            self.in_author = True
            self.current_author = ""

    def handle_data(self, data: str):
        if self.in_quote:
            self.current_quote += data
        elif self.in_author:
            self.current_author += data

    def handle_endtag(self, tag: str):
        if tag == "span" and self.in_quote:
            self.in_quote = False
        elif tag == "small" and self.in_author:
            self.in_author = False
            self.quotes.append({
                "quote": self.current_quote.strip("“ ” \" "),
                "author": self.current_author.strip()
            })

def descargar_todas_las_citas(archivo_destino: str = "citas_totales.json"):
    print("--- INICIANDO SCRAPING COMPLETO DE QUOTES TO SCRAPE ---")
    todas_las_citas = []
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    
    # Recorrer todas las páginas posibles del sitio
    for pagina in range(1, 11):
        url = f"https://toscrape.com{pagina}/"
        print(f"Descargando datos de la página {pagina}...")
        
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=10) as response:
                html_content = response.read().decode('utf-8')
            
            parser = QuotesHTMLParser()
            parser.feed(html_content)
            
            if not parser.quotes:
                # Si llegamos a una página vacía, terminamos el bucle
                break
                
            todas_las_citas.extend(parser.quotes)
            time.sleep(0.5)  # Respeto básico del servidor
            
        except Exception as e:
            print(f"Error al descargar la página {pagina}: {e}")
            break

    # Persistencia local en un archivo JSON independiente
    if todas_las_citas:
        with open(archivo_destino, "w", encoding="utf-8") as f:
            json.dump(todas_las_citas, f, ensure_ascii=False, indent=4)
        print(f"\n¡Éxito! Se han extraído {len(todas_las_citas)} citas totales y se guardaron en '{archivo_destino}'.")
    else:
        print("\nNo se pudieron extraer citas. Revisa tu conexión a red.")

if __name__ == "__main__":
    descargar_todas_las_citas()


--- INICIANDO SCRAPING COMPLETO DE QUOTES TO SCRAPE ---
Descargando datos de la página 1...
Error al descargar la página 1: <urlopen error [Errno -2] Name or service not known>

No se pudieron extraer citas. Revisa tu conexión a red.


Polemista


In [3]:
import json
import os
import sys
import logging
import random
import re
from typing import List, Dict, Optional

# Configuración limpia del logging para control de errores de lectura
logging.basicConfig(level=logging.ERROR, format="%(asctime)s - %(levelname)s - %(message)s")

class PolemistAI:
    """Motor de oratoria avanzado que genera ensayos extensos y estructurados en secciones."""
    def __init__(self, quotes_data: List[Dict[str, str]]):
        self.quotes_data = quotes_data
        self.historial_indices: List[int] = []
        
        # Diccionario semántico local para asociar términos en español con las citas en inglés
        self.mapa_conceptos = {
            "mundo": "world", "cambiar": "change", "pensar": "think", "pensamiento": "thinking",
            "elegir": "choice", "eleccion": "choices", "opcion": "choices", "habilidad": "abilities",
            "vida": "live", "vivir": "life", "milagro": "miracle", "magia": "miracle",
            "novela": "novel", "libro": "books", "literatura": "novel", "exito": "success",
            "triunfo": "success", "valor": "value", "fracaso": "fail", "fracasar": "failed",
            "verdad": "truth", "amor": "love", "tiempo": "time", "muerte": "death",
            "conocimiento": "thinking", "imaginacion": "thinking", "huevo": "world", "gallina": "world"
        }

        # Diccionario local de traducción para las citas del JSON de respaldo
        self.traducciones_citas = {
            "The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.": 
                "El mundo tal como lo hemos creado es un proceso de nuestro pensamiento. No se puede cambiar sin cambiar nuestra forma de pensar.",
            "It is our choices, Harry, that show what we truly are, far more than our abilities.": 
                "Son nuestras elecciones, Harry, las que muestran lo que realmente somos, mucho más que nuestras habilidades.",
            "There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.": 
                "Solo hay dos maneras de vivir tu vida. Una es como si nada fuera un milagro. La otra es como si todo fuera un milagro.",
            "The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.": 
                "La persona, ya sea caballero o dama, que no disfrute con una buena novela, debe ser intolerablemente estúpida.",
            "Try not to become a man of success. Rather become a man of value.": 
                "Intenta no convertirte en un hombre de éxito. Más bien conviértete en un hombre de valor.",
            "I have not failed. I've just found 10,000 ways that won't work.": 
                "No he fracasado. Solo he encontrado 10,000 maneras que no funcionan.",
            "A day without sunshine is like, you know, night.": 
                "Un día sin sol es como, ya sabes, la noche.",
            "A woman is like a tea bag; you never know how strong it is until it's in hot water.": 
                "Una mujer es como una bolsa de té; nunca sabes lo fuerte que es hasta que está en agua caliente.",
            "It is never too late to be what you might have been.": 
                "Nunca es demasiado tarde para ser lo que podrías haber sido.",
            "Life is what happens to us while we are making other plans.": 
                "La vida es lo que nos pasa mientras hacemos otros planes."
        }

        # SECCIÓN I: INTRODUCCIÓN (1 Párrafo)
        self.intro_templates = [
            "El debate intelectual en torno a '{pregunta}' constituye una de las discusiones más profundas de la ontología contemporánea. A lo largo de la historia del pensamiento, esta disyuntiva ha desafiado las certezas cotidianas, forzando a filósofos y científicos a examinar los cimientos de la percepción humana. En este contexto de crisis epistemológica, resulta imperativo explorar cómo las estructuras internas de la lógica modelan nuestras conclusiones. Por lo tanto, este ensayo analizará de manera rigurosa las implicaciones teóricas del problema, destacando la necesidad de adoptar un marco crítico integral para resolver la paradoja implícita en su formulación."
        ]
        
        # SECCIÓN II: DESARROLLO (4 Párrafos)
        self.desarrollo_p1 = [
            "Para comprender la dimensión exacta de este fenómeno, es necesario segmentar sus componentes fundamentales. En primer lugar, la aproximación abstracta al problema nos revela que los juicios de valor preconcebidos tienden a sesgar la búsqueda de una respuesta objetiva. Cuando nos enfrentamos a nociones complejas, la mente humana busca regularidades semánticas inmediatas, ignorando a menudo que el tejido mismo de la duda planteada requiere un examen metodológico mucho más sutil y microscópico."
        ]
        
        self.desarrollo_p2 = [
            "Es precisamente en esta encrucijada teórica donde la agudeza intelectual de {autor} adquiere una relevancia monumental. Dentro del repositorio documental analizado, el autor establece una premisa ineludible que redefine el debate: {cita_bloque}. A través de estas palabras exactas, se nos recuerda de manera contundente que los límites de nuestra comprensión conceptual están determinados por el marco ideológico que decidimos validar de antemano."
        ]
        
        self.desarrollo_p3 = [
            "Además, un análisis pormenorizado del entorno operativo nos obliga a contrastar esta postura con las corrientes opositoras tradicionales. Cuando los teóricos contemporáneos intentan formular soluciones mecanicistas o puramente empiristas a su pregunta, chocan inevitablemente contra la barrera lógica propuesta en la cita anterior. La argumentación sólida demuestra que no podemos resolver un dilema utilizando el mismo nivel de pensamiento que lo originó."
        ]
        
        self.desarrollo_p4 = [
            "Por último, este enfoque sistemático promueve una reevaluación del autoconcepto intelectual de quien interroga. Al alcanzar metas de comprensión crítica y superar los sesgos cognitivos del lenguaje común, el investigador se encuentra en una posición metodológica mucho más robusta y esclarecedora. Esto tiene un impacto directamente proporcional en la validez del debate, transformando una simple pregunta en un motor de cambio epistemológico duradero."
        ]

        # SECCIÓN III: CONCLUSIÓN (1 Párrafo)
        self.conclusion_templates = [
            "En resumen, el cuestionamiento sobre '{pregunta}' no es simplemente un ejercicio de retórica vacía, sino un desafío que impacta directamente en las bases del saber. A través del desglose conceptual realizado, la deconstrucción de los argumentos opuestos y el blindaje analítico proporcionado por las tesis de {autor}, la discusión se convierte en una herramienta invaluable para expandir los horizontes de la razón. Como sociedad intelectualmente activa, debemos promover de forma concisa la adopción de estas metodologías de análisis crítico en nuestra rutina académica, manteniendo nuestra mente en óptimas condiciones para enfrentar las futuras paradojas de la existencia."
        ]

    def _buscar_cita_por_concepto(self, query: str) -> Dict[str, str]:
        """Busca coincidencias conceptuales o rota obligatoriamente en la base de datos local."""
        query_limpia = re.sub(r'[^\w\s]', '', query.lower())
        palabras_usuario = query_limpia.split()

        for palabra in palabras_usuario:
            if palabra in self.mapa_conceptos:
                concepto_en = self.mapa_conceptos[palabra]
                for idx, item in enumerate(self.quotes_data):
                    if concepto_en in item['quote'].lower() and idx not in self.historial_indices:
                        self.historial_indices.append(idx)
                        return item

        indices_disponibles = [i for i in range(len(self.quotes_data)) if i not in self.historial_indices]
        if not indices_disponibles:
            self.historial_indices.clear()
            indices_disponibles = list(range(len(self.quotes_data)))
            
        elegido_idx = random.choice(indices_disponibles)
        self.historial_indices.append(elegido_idx)
        return self.quotes_data[elegido_idx]

    def generar_ensayo_largo(self, query: str) -> str:
        """Construye un ensayo de 6 párrafos siguiendo fielmente el modelo formal de la imagen."""
        # Sanitizar entrada eliminando espacios extraños y saltos de línea ocultos
        query_sanitizada = " ".join(query.strip().split())
        if not query_sanitizada:
            return "[Polemista] Error: Ingrese un cuestionamiento válido y visible."

        cita_item = self._buscar_cita_por_concepto(query_sanitizada)
        autor = cita_item["author"]
        cita_original = cita_item["quote"]
        
        # Obtener traducción local si existe en nuestro mapa, sino dejar solo la original
        traduccion_es = self.traducciones_citas.get(cita_original, "")
        if traduccion_es:
            cita_bloque = f'"{cita_original}" (Traducido al español: "{traduccion_es}")'
        else:
            cita_bloque = f'"{cita_original}"'

        pregunta_limpia = query_sanitizada.strip("¿? ‽ ")

        # Construcción independiente de cada una de las secciones
        introduccion = random.choice(self.intro_templates).format(pregunta=pregunta_limpia)
        
        dp1 = random.choice(self.desarrollo_p1)
        dp2 = random.choice(self.desarrollo_p2).format(autor=autor, cita_bloque=cita_bloque)
        dp3 = random.choice(self.desarrollo_p3)
        dp4 = random.choice(self.desarrollo_p4)
        desarrollo = f"{dp1}\n\n{dp2}\n\n{dp3}\n\n{dp4}"
        
        conclusion = random.choice(self.conclusion_templates).format(pregunta=pregunta_limpia, autor=autor)

        return (
            f"[INTRODUCCIÓN]\n{introduccion}\n\n"
            f"[DESARROLLO]\n{desarrollo}\n\n"
            f"[CONCLUSIÓN]\n{conclusion}")


def cargar_base_datos(filepath: str) -> Optional[List[Dict[str, str]]]:
    """Carga los datos estrictamente desde el almacenamiento local JSON."""
    if not os.path.exists(filepath):
        return None
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, IOError):
        return None

def main():
    archivo_db = "citas_totales.json"
    print("--- INICIALIZANDO POLEMISTA EXPERTO (MODELO ACADÉMICO EXTENSO V2) ---")
    
    quotes = cargar_base_datos(archivo_db)
    if not quotes:
        print(f"\n[Error Crítico] No se detectó la base de datos '{archivo_db}'.")
        print("Por favor, vuelve a ejecutar el comando de inyección rápida de python3 para crearla.")
        sys.exit(1)
        
    polemist = PolemistAI(quotes)
    print(f"\n¡Base de datos cargada! Ensayos largos de 6 párrafos con traducción local integrados.")
    print("Escribe 'salir' para finalizar.\n")
    
    while True:
        try:
            user_input = input("Haz una pregunta compleja o filosófica:\n> ")
            
            # Si el usuario presiona enter por error sin escribir nada, continuar el ciclo sin romperlo
            if not user_input.strip():
                continue
                
            if user_input.strip().lower() in ["salir", "exit", "quit"]:
                print("\n[Polemista] El debate formal ha concluido exitosamente.")
                break
                
            ensayo_resultado = polemist.generar_ensayo_largo(user_input)
            
            print(f"\n{'='*25} ENSAYO ACADÉMICO EXTENSO {'='*25}\n")
            print(ensayo_resultado)
            print(f"\n{'='*78}\n")
            
        except (KeyboardInterrupt, EOFError):
            print("\n\n[Polemista] Proceso interactivo finalizado desde la terminal.")
            break

if __name__ == "__main__":
    main()


--- INICIALIZANDO POLEMISTA EXPERTO (MODELO ACADÉMICO EXTENSO V2) ---

¡Base de datos cargada! Ensayos largos de 6 párrafos con traducción local integrados.
Escribe 'salir' para finalizar.


========================= ENSAYO ACADÉMICO EXTENSO =========================

[INTRODUCCIÓN]
El debate intelectual en torno a 'La masturbación es mala' constituye una de las discusiones más profundas de la ontología contemporánea. A lo largo de la historia del pensamiento, esta disyuntiva ha desafiado las certezas cotidianas, forzando a filósofos y científicos a examinar los cimientos de la percepción humana. En este contexto de crisis epistemológica, resulta imperativo explorar cómo las estructuras internas de la lógica modelan nuestras conclusiones. Por lo tanto, este ensayo analizará de manera rigurosa las implicaciones teóricas del problema, destacando la necesidad de adoptar un marco crítico integral para resolver la paradoja implícita en su formulación.

[DESARROLLO]
Para comprender la dimens